# Tau Supersymmetry Anomaly Detection — Analysis Overview

<div style="text-align: justify">

This notebook serves as a guide to the <b>Tau Supersymmetry Anomaly Detection</b> analysis. The analysis uses <b>autoencoders (AE)</b> and <b>variational autoencoders (VAE)</b> to detect anomalous SUSY signals in ATLAS tau data via unsupervised anomaly detection. Models are trained on <b>background only</b> — signal events appear as anomalies with high reconstruction error. Each stage is available both as an <b>interactive Jupyter notebook</b> (with inline plots and outputs) and as a <b>CLI command</b> via a single Hydra-dispatched entry point (`run.py`).

</div>

## Pipeline Overview

The diagram below shows how the stages connect. Each notebook can be explored independently — outputs from previous stages are persisted as Parquet files.

```
ROOT ntuples
     │
     ▼
┌─────────────────────────────────┐
│  01  Preprocessing              │  ──▶  Selection cuts, event weights, ROOT → Parquet
└─────────────────┬───────────────┘
                  ▼
┌─────────────────────────────────┐
│  02  Feature Engineering        │  ──▶  Physics features, sample merging, rectangularization
└─────────────────┬───────────────┘
                  ▼
┌─────────────────────────────────┐
│  03  Exploratory Data Analysis  │  ──▶  Data quality, sample composition, correlations, feature distributions
└─────────────────┬───────────────┘
                  ▼
┌─────────────────────────────────┐
│  04  Hyperparameter Tuning      │  ──▶  Ray Tune search with ASHA scheduler
└─────────────────┬───────────────┘
                  ▼
┌─────────────────────────────────┐
│  05a Training (AE)              │  ──▶  Background-only autoencoder training with PyTorch Lightning
│  05b Training (VAE)             │  ──▶  Background-only VAE training with loss decomposition monitoring
└─────────────────┬───────────────┘
                  ▼
┌─────────────────────────────────┐
│  06a Evaluation (AE)            │  ──▶  Anomaly scoring, ROC AUC, SIC curves, reconstruction diagnostics, latent space
│  06b Evaluation (VAE)           │  ──▶  Anomaly scoring, ROC AUC, SIC curves, reconstruction diagnostics, VAE latent analysis
└─────────────────┬───────────────┘
                  ▼
┌─────────────────────────────────┐
│  Serving (FastAPI)              │  ──▶  REST API for real-time anomaly scoring
└─────────────────────────────────┘
```

## Notebooks

| # | Notebook | Description | CLI Equivalent |
|---|----------|-------------|----------------|
| 01 | [Preprocessing](./01_preprocessing.ipynb) | Reads ROOT ntuples, applies selection cuts, computes event weights, and exports to Parquet | `uv run python run.py stage=preprocess` |
| 02 | [Feature Engineering](./02_feature_engineering.ipynb) | Prepares physics-based input features, merges samples, and rectangularizes into a single DataFrame | `uv run python run.py stage=feature_engineer` |
| 03 | [Exploratory Data Analysis](./03_eda.ipynb) | Examines data quality, sample composition, feature correlations, and feature distributions | `uv run python run.py stage=eda` |
| 04 | [Hyperparameter Tuning](./04_hyperparameter_tuning.ipynb) | Searches hyperparameter space using Ray Tune with ASHA scheduler | `uv run python run.py stage=tune` |
| 05a | [AE Training](./05a_ae_training.ipynb) | Trains Autoencoder on background-only data with early stopping and WandB tracking | `uv run python run.py stage=train model=ae` |
| 05b | [VAE Training](./05b_vae_training.ipynb) | Trains VAE on background-only data with loss decomposition monitoring | `uv run python run.py stage=train model=vae` |
| 06a | [AE Evaluation](./06a_ae_evaluation.ipynb) | Evaluates AE: ROC AUC, SIC curves, reconstruction diagnostics, latent space analysis | `uv run python run.py stage=evaluate model=ae` |
| 06b | [VAE Evaluation](./06b_vae_evaluation.ipynb) | Evaluates VAE: ROC AUC, SIC curves, reconstruction diagnostics, VAE latent analysis | `uv run python run.py stage=evaluate model=vae` |

## CLI Pipeline

The entire analysis can be run from the command line without opening any notebooks. All stages are dispatched through a single Hydra entry point (`run.py`) and configured via YAML files in `configs/`.

```bash
# Run individual stages
uv run python run.py stage=preprocess          # 01 — ROOT → Parquet
uv run python run.py stage=feature_engineer    # 02 — feature extraction & sample merging
uv run python run.py stage=eda                 # 03 — exploratory data analysis
uv run python run.py stage=tune                # 04 — hyperparameter tuning (Ray Tune)
uv run python run.py stage=train model=ae      # 05 — AE training
uv run python run.py stage=train model=vae     # 05 — VAE training
uv run python run.py stage=evaluate model=ae   # 06 — AE evaluation
uv run python run.py stage=evaluate model=vae  # 06 — VAE evaluation
```

All hyperparameters can be overridden via the command line:

```bash
uv run python run.py stage=train model=ae model.latent_dim=16 model.learning_rate=0.001
uv run python run.py stage=tune tuning.num_samples=200
```

## Model Serving

After training, models can be deployed as a REST API using FastAPI. The server loads a trained AE or VAE checkpoint and serves anomaly scores over HTTP.

```bash
# Start inference server for a trained AE
uv run python run.py stage=serve model=ae

# Start inference server for a trained VAE
uv run python run.py stage=serve model=vae
```

| Endpoint | Method | Description |
|----------|--------|-------------|
| `/health` | GET | Health check — is the server running? |
| `/model/info` | GET | Model metadata (expected features, model type) |
| `/predict` | POST | Score a single event (returns anomaly score) |
| `/predict/batch` | POST | Score multiple events in one request |

```bash
# Example: score a single event
curl -X POST http://localhost:8000/predict \
  -H "Content-Type: application/json" \
  -d '{"features": {"met": 250.0, "jet_n": 3, "dphi": 1.2}}'

# Interactive API docs (auto-generated by FastAPI)
open http://localhost:8000/docs
```

The serving layer uses an **adapter pattern** (`src/serving/registry.py`) so that AE and VAE models share a uniform interface — the API code never touches model-specific logic.

## Stack

| Category | Libraries |
|----------|----------|
| Data Processing | uproot, awkward-array, NumPy, SciPy, pandas, Pandera |
| Visualisation | Matplotlib, Seaborn |
| Machine Learning | PyTorch, PyTorch Lightning, torchmetrics, torcheval, scikit-learn, Ray Tune |
| Infrastructure | Docker, uv, GitHub Actions |
| Configuration & Tracking | Hydra, OmegaConf, Weights & Biases |
| Serving / API | FastAPI, Uvicorn, Pydantic |
| Testing & QA | pytest, pytest-cov, mypy, Ruff, pre-commit |
| Physics | atlas-mpl-style |